In [ ]:
## Load functions
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression


import joblib
import json


In [ ]:
# Helper functions for regularized logistic regression and diagnostics

def build_model_form(target, features):
    return f"{target} = " + " + ".join(features)


def ks_statistic(y_true, y_score):
    temp = pd.DataFrame({
        "y_true": y_true,
        "y_score": y_score
    }).sort_values("y_score", ascending=False)

    temp["good"] = (temp["y_true"] == 0).astype(int)
    temp["bad"] = (temp["y_true"] == 1).astype(int)

    temp["cum_good"] = temp["good"].cumsum() / temp["good"].sum()
    temp["cum_bad"] = temp["bad"].cumsum() / temp["bad"].sum()

    return (temp["cum_bad"] - temp["cum_good"]).abs().max()


def calculate_classification_diagnostics(
    y_train,
    p_train,
    y_val,
    p_val,
    threshold=0.50
):
    def _metrics(y_true, y_score, dataset_name):
        y_pred = (y_score >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        return {
            "dataset": dataset_name,
            "threshold": threshold,
            "auc": roc_auc_score(y_true, y_score),
            "gini": 2 * roc_auc_score(y_true, y_score) - 1,
            "ks": ks_statistic(y_true, y_score),
            "pr_auc": average_precision_score(y_true, y_score),
            "log_loss": log_loss(y_true, y_score),
            "brier_score": brier_score_loss(y_true, y_score),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        }

    return pd.DataFrame([
        _metrics(y_train, p_train, "train"),
        _metrics(y_val, p_val, "validation")
    ])


def extract_sklearn_coefficients(model, features):
    coef_df = pd.DataFrame({
        "variable": features,
        "coefficient": model.coef_[0],
        "odds_ratio": np.exp(model.coef_[0])
    })

    coef_df = coef_df.sort_values(
        "coefficient",
        ascending=False
    ).reset_index(drop=True)

    return coef_df

# Main Function: Run Regularized Logistic Regression

def run_regularized_logistic_regression(
    model_registry,
    X_train,
    X_val,
    y_train,
    y_val,
    model_number,
    model_name,
    target,
    features,
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=2000,
    threshold=0.50,
    analyst_comments="",
    display_outputs=True
):
    missing_features = [f for f in features if f not in X_train.columns]
    if missing_features:
        raise ValueError(f"Missing features: {missing_features}")
    if model_number in model_registry:
        raise ValueError(f"{model_number} already exists.")

    X_train_model = X_train[features].copy()
    X_val_model = X_val[features].copy()

    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver=solver,
        max_iter=max_iter,
        random_state=42
    )

    model.fit(X_train_model, y_train)

    p_train = model.predict_proba(X_train_model)[:, 1]
    p_val = model.predict_proba(X_val_model)[:, 1]

    diagnostics_table = calculate_classification_diagnostics(
        y_train=y_train,
        p_train=p_train,
        y_val=y_val,
        p_val=p_val,
        threshold=threshold
    )

    coefficient_table = extract_sklearn_coefficients(
        model=model,
        features=features
    )
    model_form = build_model_form(target, features)
    metadata = {
        "model_number": model_number,
        "model_name": model_name,
        "model_form": model_form,
        "features": features,
        "num_features": len(features),
        "penalty": penalty,
        "C": C,
        "solver": solver,
        "threshold": threshold,
        "intercept": model.intercept_[0],
        "analyst_comments": analyst_comments
    }
    model_registry[model_number] = {
        "metadata": metadata,
        "model": model,
        "coefficients": coefficient_table,
        "diagnostics": diagnostics_table
    }

    if display_outputs:
        print("=" * 100)
        print(f"{model_number}: {model_name}")
        print("=" * 100)
        print("\nMODEL FORM:")
        print(model_form)
        print("\nMODEL SETTINGS:")
        print(f"Penalty: {penalty}")
        print(f"C: {C}")
        print(f"Solver: {solver}")
        print("\nCOEFFICIENTS:")
        display(coefficient_table)
        print("\nDIAGNOSTICS:")
        display(diagnostics_table)
        if analyst_comments:
            print("\nANALYST COMMENTS:")
            print(analyst_comments)

    return model_registry

def export_model_registry_to_excel(
    model_registry,
    file_path
):
    with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
        summary_rows = []
        for model_num, model_data in model_registry.items():
            meta = model_data.get("metadata", {})
            diag = model_data.get("diagnostics", pd.DataFrame()).copy()
            row = {
                "model_number": model_num,
                "model_name": meta.get("model_name", ""),
                "num_features": meta.get("num_features", ""),
                "threshold": meta.get("threshold", ""),
                "comments": meta.get("analyst_comments", "")
            }
            # Optional metadata fields
            optional_fields = [
                "aic",
                "bic",
                "mcfadden_r2",
                "penalty",
                "C",
                "solver"
            ]
            for field in optional_fields:
                row[field] = meta.get(field, "")

            if not diag.empty and "dataset" in diag.columns:
                train_rows = diag.loc[diag["dataset"] == "train"]
                val_rows = diag.loc[diag["dataset"] == "validation"]

                if not train_rows.empty:
                    train_row = train_rows.iloc[0]
                    row["train_auc"] = train_row.get("auc", "")
                    row["train_ks"] = train_row.get("ks", "")
                    row["train_log_loss"] = train_row.get("log_loss", "")
                    row["train_brier"] = train_row.get("brier_score", "")

                if not val_rows.empty:
                    val_row = val_rows.iloc[0]
                    row["val_auc"] = val_row.get("auc", "")
                    row["val_ks"] = val_row.get("ks", "")
                    row["val_log_loss"] = val_row.get("log_loss", "")
                    row["val_brier"] = val_row.get("brier_score", "")

            summary_rows.append(row)

        summary_df = pd.DataFrame(summary_rows)
        summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )
        for model_num, model_data in model_registry.items():
            start_row = 0
            meta = model_data.get("metadata", {})
            meta_df = pd.DataFrame({
                "field": list(meta.keys()),
                "value": list(meta.values())
            })
            meta_df.to_excel(
                writer,
                sheet_name=model_num[:31],
                startrow=start_row,
                index=False
            )

            start_row += len(meta_df) + 3
            coef_df = model_data.get("coefficients", pd.DataFrame())
            if not coef_df.empty:
                coef_df.to_excel(
                    writer,
                    sheet_name=model_num[:31],
                    startrow=start_row,
                    index=False
                )

                start_row += len(coef_df) + 3
            diag_df = model_data.get("diagnostics", pd.DataFrame())
            if not diag_df.empty:
                diag_df.to_excel(
                    writer,
                    sheet_name=model_num[:31],
                    startrow=start_row,
                    index=False
                )

                start_row += len(diag_df) + 3
            vif_df = model_data.get("vif", pd.DataFrame())
            if not vif_df.empty:
                vif_df.to_excel(
                    writer,
                    sheet_name=model_num[:31],
                    startrow=start_row,
                    index=False
                )
                start_row += len(vif_df) + 3
    print("Saved:", file_path)

def save_model_artifact(model_registry, model_id, model_dir, config_dir):
    record = model_registry[model_id]

    model_path = model_dir / f"{model_id}.joblib"
    config_path = config_dir / f"{model_id}_config.json"

    joblib.dump(record["model"], model_path)

    metadata = record.get("metadata", {}).copy()

    config = {
        "model_id": model_id,
        "model_class": str(type(record["model"]).__name__),
        "metadata": metadata,
        "features": metadata.get("features", None),
        "num_features": metadata.get("num_features", None),
    }

    if "diagnostics" in record:
        try:
            config["diagnostics_preview"] = record["diagnostics"].to_dict(orient="records")
        except Exception:
            pass

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4, default=str)

    print("Saved:", model_path)
    print("Saved:", config_path)

In [17]:
# Project directory setup

CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_DIR = OUTPUT_DIR / "saved_models"
CONFIG_DIR = OUTPUT_DIR / "model_configs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project root: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab
Data directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/data
Output directory: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs


In [18]:
# Load dataset

df_scaled = pd.read_parquet(DATA_DIR / "df_scaled_model_ready.parquet")
print(f"Data shape: {df_scaled.shape}")
print(df_scaled.head())
print(df_scaled.dtypes)

Data shape: (149390, 16)
   SeriousDlqin2yrs       age    age_sq  NumberOfTime30-59DaysPastDueNotWorse  \
0                 1 -0.496191 -0.578484                              0.416854   
1                 0 -0.835742 -0.843467                             -0.102229   
2                 0 -0.971562 -0.940732                              0.157313   
3                 0 -1.514844 -1.279910                             -0.102229   
4                 0 -0.224551 -0.344051                              0.157313   

   NumberOfTime60-89DaysPastDueNotWorse  NumberOfTimes90DaysLate  \
0                             -0.055768                -0.062235   
1                             -0.055768                -0.062235   
2                             -0.055768                 0.199123   
3                             -0.055768                -0.062235   
4                             -0.055768                -0.062235   

   NumberOfOpenCreditLinesAndLoans  NumberRealEstateLoansOrLines  \
0          

In [19]:
# Create test and validation sets

target = "SeriousDlqin2yrs"

candidate_features = [
    col for col in df_scaled.columns
    if col != target
]

X = df_scaled[candidate_features]
y = df_scaled[target]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Number of candidate features:", len(candidate_features))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("Train bad rate:", y_train.mean())
print("Validation bad rate:", y_val.mean())



Number of candidate features: 15
X_train: (104573, 15)
X_val: (44817, 15)
Train bad rate: 0.06699626098514913
Validation bad rate: 0.06700582368297744


In [20]:
# Initialize model registry to store models, coefficients, diagnostics, and metadata
model_registry = {}

In [22]:
# Model 01
model_registry = run_regularized_logistic_regression(
    model_registry=model_registry,
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    model_number="R001",
    model_name="Regularized Logistic - L2 Baseline",
    target=target,
    features=candidate_features,
    penalty="l2",
    C=1.0,
    solver="liblinear",
    analyst_comments="Baseline regularized logistic using all scaled features."
)

/Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


R001: Regularized Logistic - L2 Baseline

MODEL FORM:
SeriousDlqin2yrs = age + age_sq + NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate + NumberOfOpenCreditLinesAndLoans + NumberRealEstateLoansOrLines + NumberOfDependents_median + NumberOfDependents_missing_flag + MonthlyIncome_median + MonthlyIncome_missing_flag + DebtRatio_log + DebtRatio_high_flag + RevolvingUtilization_log + RevolvingUtilization_high_flag

MODEL SETTINGS:
Penalty: l2
C: 1.0
Solver: liblinear

COEFFICIENTS:


,variable,coefficient,odds_ratio
0,NumberOfTime30-59DaysPastDueNotWorse,1.156770,3.179645
1,NumberOfTimes90DaysLate,1.108822,3.030785
2,RevolvingUtilization_log,0.936848,2.551926
3,DebtRatio_log,0.253124,1.288043
4,NumberOfOpenCreditLinesAndLoans,0.110743,1.117108
5,age,0.104327,1.109963
6,MonthlyIncome_missing_flag,0.088298,1.092314
7,NumberOfDependents_median,0.063111,1.065145
8,NumberRealEstateLoansOrLines,0.043493,1.044453
9,NumberOfDependents_missing_flag,0.000828,1.000829



DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.803332,0.606663,0.472734,0.292666,0.202738,0.054615,0.934180,0.605128,0.050528,0.093268,97336,231,6652,354
1,validation,0.5,0.805867,0.611733,0.474014,0.294274,0.202110,0.054530,0.934221,0.611336,0.050283,0.092923,41718,96,2852,151



ANALYST COMMENTS:
Baseline regularized logistic using all scaled features.


In [11]:
# Model 02
model_registry = run_regularized_logistic_regression(
    model_registry=model_registry,
    X_train=X_train,
    X_val=X_val,
    y_train=y_train,
    y_val=y_val,
    model_number="R002",
    model_name="Regularized Logistic - L1 Baseline",
    target=target,
    features=candidate_features,
    penalty="l1",
    C=1.0,
    solver="liblinear",
    analyst_comments="Baseline regularized logistic using all scaled features."
)

/Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


R002: Regularized Logistic - L1 Baseline

MODEL FORM:
SeriousDlqin2yrs = age + age_sq + NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate + NumberOfOpenCreditLinesAndLoans + NumberRealEstateLoansOrLines + NumberOfDependents_median + NumberOfDependents_missing_flag + MonthlyIncome_median + MonthlyIncome_missing_flag + DebtRatio_log + DebtRatio_high_flag + RevolvingUtilization_log + RevolvingUtilization_high_flag

MODEL SETTINGS:
Penalty: l1
C: 1.0
Solver: liblinear

COEFFICIENTS:


,variable,coefficient,odds_ratio
0,NumberOfTime30-59DaysPastDueNotWorse,1.159954,3.189788
1,NumberOfTimes90DaysLate,1.112941,3.043295
2,RevolvingUtilization_log,0.937018,2.552358
3,DebtRatio_log,0.251534,1.285996
4,NumberOfOpenCreditLinesAndLoans,0.110799,1.117170
5,age,0.090077,1.094258
6,MonthlyIncome_missing_flag,0.086532,1.090386
7,NumberOfDependents_median,0.063264,1.065308
8,NumberRealEstateLoansOrLines,0.043672,1.044640
9,NumberOfDependents_missing_flag,0.000530,1.000530



DIAGNOSTICS:


,dataset,threshold,auc,gini,ks,pr_auc,log_loss,brier_score,accuracy,precision,recall,f1,true_negative,false_positive,false_negative,true_positive
0,train,0.5,0.803322,0.606644,0.472569,0.292657,0.202738,0.054614,0.934180,0.605128,0.050528,0.093268,97336,231,6652,354
1,validation,0.5,0.805853,0.611705,0.474464,0.294302,0.202105,0.054527,0.934244,0.612903,0.050616,0.093510,41718,96,2851,152



ANALYST COMMENTS:
Baseline regularized logistic using all scaled features.


In [14]:
# Save both models to excel for easy comparison and documentation

excel_path = OUTPUT_DIR / "regularized_logistic_model_registry.xlsx"

export_model_registry_to_excel(
    model_registry=model_registry,
    file_path=excel_path
)

Saved: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/regularized_logistic_model_registry.xlsx


In [23]:
save_model_artifact(
    model_registry=model_registry,
    model_id="R001",  # replace with best regularized model
    model_dir=MODEL_DIR,
    config_dir=CONFIG_DIR
)

Saved model: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/saved_models/R001.joblib
Saved config: /Users/sumanchattopadhyay/Documents/Documents - Suman’s MacBook Air/Projects/credit-risk-mlops-lab/outputs/model_configs/R001_config.json
